# VeriFact — Fake News Detector
## Notebook 4: Gemini API Integration
**Goal:** Use Gemini to (1) explain WHY an article is fake/real, (2) handle non-English articles, (3) provide a student-friendly summary.

> **API Key:** Store your Gemini API key in Colab Secrets (key icon on left sidebar) as `GEMINI_API_KEY`

---

## Step 0 — Setup

In [ ]:
from google.colab import drive, userdata
drive.mount('/content/drive')

# Get API key from Colab Secrets
GEMINI_API_KEY = userdata.get('GEMINI_API_KEY')
print("API key loaded:", "Yes" if GEMINI_API_KEY else "NOT FOUND — add it to Colab Secrets!")

In [ ]:
!pip install google-generativeai langdetect --quiet
print("Done.")

In [ ]:
import google.generativeai as genai
import pickle
import json
import re
import time
import warnings
warnings.filterwarnings('ignore')

# Configure Gemini
genai.configure(api_key=GEMINI_API_KEY)
model = genai.GenerativeModel('gemini-1.5-flash')  # free tier model

print("Gemini configured. Model: gemini-1.5-flash")

## Step 1 — Gemini Prompt Engineering

Three separate prompts for three use cases:
1. **Explainability** — why did the ML model give this verdict?
2. **Non-English fallback** — full analysis when ML model can't handle it
3. **Low confidence fallback** — when ML model is uncertain

In [ ]:
def build_explanation_prompt(article_text, ml_verdict, ml_confidence, tone_score, sentiment):
    """
    Prompt for explainability — called after ML model gives a verdict.
    Gemini explains WHY in student-friendly language.
    """
    return f"""
You are VeriFact, an AI fact-checking assistant designed to help students identify fake news.

A machine learning model analyzed this news article and gave the following verdict:
- Verdict: {ml_verdict}
- Confidence: {ml_confidence:.1f}%
- Sensationalism tone score: {tone_score}/100 (0=calm, 100=very sensational)
- Sentiment: {sentiment['label']} (score: {sentiment['compound']})

Article text (first 800 characters):
\"\"\"{article_text[:800]}\"\"\"

Your task: Write a SHORT, student-friendly explanation (3-4 sentences max) that:
1. Confirms the verdict in plain language
2. Points out 2-3 specific signals (language patterns, tone, missing sources, etc.)
3. Gives one tip on how to verify this type of article

Keep the tone helpful and educational, not alarming. Respond in English only.
Do NOT repeat the verdict label — just explain the reasoning.
"""


def build_multilingual_prompt(article_text):
    """
    Prompt for non-English articles — Gemini does full analysis.
    Returns structured JSON response.
    """
    return f"""
You are VeriFact, a multilingual AI fact-checking assistant for students.

Analyze the following news article and determine if it is likely Fake or Real news.
The article may be in any language — analyze it in its original language.

Article:
\"\"\"{article_text[:1200]}\"\"\"

Respond ONLY with a valid JSON object (no markdown, no backticks):
{{
  "verdict": "Fake" or "Real",
  "confidence": number between 0 and 100,
  "explanation": "2-3 sentence student-friendly explanation in English",
  "signals": ["signal 1", "signal 2", "signal 3"],
  "verify_tip": "one actionable tip to verify this article"
}}
"""


def build_low_confidence_prompt(article_text, ml_verdict, ml_confidence):
    """
    Prompt for low-confidence ML predictions.
    Gemini provides a second opinion.
    """
    return f"""
You are VeriFact, an AI fact-checking assistant.

A machine learning model was uncertain about this article (confidence: {ml_confidence:.1f}%, verdict: {ml_verdict}).
Provide your independent assessment.

Article:
\"\"\"{article_text[:1000]}\"\"\"

Respond ONLY with a valid JSON object (no markdown, no backticks):
{{
  "verdict": "Fake" or "Real" or "Uncertain",
  "confidence": number between 0 and 100,
  "explanation": "2-3 sentence explanation in simple English",
  "signals": ["signal 1", "signal 2"],
  "verify_tip": "one specific tip to verify"
}}
"""


print("Prompt functions defined.")

## Step 2 — Gemini API Call Wrapper

In [ ]:
def call_gemini(prompt, expect_json=False, retries=2):
    """
    Safe wrapper around Gemini API calls.
    Handles rate limits, JSON parsing, and errors gracefully.
    """
    for attempt in range(retries + 1):
        try:
            response = model.generate_content(
                prompt,
                generation_config=genai.types.GenerationConfig(
                    temperature=0.2,     # low temp for factual consistency
                    max_output_tokens=400
                )
            )
            text = response.text.strip()

            if expect_json:
                # Clean any markdown code fences if Gemini adds them
                text = re.sub(r'```json|```', '', text).strip()
                return json.loads(text)
            return text

        except json.JSONDecodeError:
            if attempt < retries:
                time.sleep(1)
                continue
            return {
                'verdict': 'Uncertain',
                'confidence': 50,
                'explanation': 'Analysis inconclusive. Please verify with a trusted source.',
                'signals': ['Could not parse response'],
                'verify_tip': 'Check multiple reputable news sources.'
            }
        except Exception as e:
            if attempt < retries:
                time.sleep(2)
                continue
            error_msg = str(e)
            if 'quota' in error_msg.lower() or '429' in error_msg:
                return {'error': 'rate_limit', 'explanation': 'API rate limit reached. Try again in a moment.'}
            return {'error': str(e), 'explanation': 'Gemini analysis unavailable.'}


print("Gemini wrapper defined.")

## Step 3 — Master Prediction Function

This is the function the Streamlit app will call.

In [ ]:
import sys
sys.path.append('/content/drive/MyDrive/VeriFact/utils/')

import importlib.util, types

# Load nlp_utils from Drive
spec = importlib.util.spec_from_file_location(
    'nlp_utils', '/content/drive/MyDrive/VeriFact/utils/nlp_utils.py')
nlp_utils = importlib.util.module_from_spec(spec)
spec.loader.exec_module(nlp_utils)

# Load ML model
with open('/content/drive/MyDrive/VeriFact/models/best_model.pkl', 'rb') as f:
    ml_pipeline = pickle.load(f)

with open('/content/drive/MyDrive/VeriFact/models/model_metadata.json', 'r') as f:
    metadata = json.load(f)

CONFIDENCE_THRESHOLD = metadata.get('confidence_threshold', 0.75)

import nltk
nltk.download('stopwords', quiet=True)
from nltk.corpus import stopwords
STOP_WORDS = set(stopwords.words('english'))

def clean_text(text):
    text = str(text).lower()
    text = re.sub(r'http\S+|www\.\S+', '', text)
    text = re.sub(r'[^a-z\s]', '', text)
    text = re.sub(r'\s+', ' ', text).strip()
    tokens = [t for t in text.split() if t not in STOP_WORDS and len(t) > 2]
    return ' '.join(tokens)


def full_predict(article_text):
    """
    Master prediction function.
    Returns a full result dict for the Streamlit app.
    
    Flow:
      1. NLP enrichment (language, sentiment, tone, NER)
      2. Language check → if non-English → Gemini full analysis
      3. If English → ML model → if low confidence → Gemini fallback
      4. If high confidence → Gemini explanation only
    """
    result = {
        'original_text': article_text,
        'ml_used':       False,
        'gemini_used':   False,
        'verdict':       None,
        'confidence':    None,
        'explanation':   None,
        'signals':       [],
        'verify_tip':    None,
        'nlp':           None
    }

    # Step 1 — NLP enrichment
    nlp_data = nlp_utils.analyze_article(article_text)
    result['nlp'] = nlp_data

    # Step 2 — Non-English → Gemini full analysis
    if not nlp_data['is_english']:
        print(f"[INFO] Non-English detected ({nlp_data['language']}). Routing to Gemini.")
        gemini_result = call_gemini(
            build_multilingual_prompt(article_text),
            expect_json=True
        )
        result.update({
            'gemini_used': True,
            'verdict':     gemini_result.get('verdict', 'Uncertain'),
            'confidence':  gemini_result.get('confidence', 50),
            'explanation': gemini_result.get('explanation', ''),
            'signals':     gemini_result.get('signals', []),
            'verify_tip':  gemini_result.get('verify_tip', '')
        })
        return result

    # Step 3 — English → ML model
    cleaned = clean_text(article_text)
    clf = ml_pipeline.named_steps['clf']

    if hasattr(clf, 'predict_proba'):
        proba      = ml_pipeline.predict_proba([cleaned])[0]
        confidence = float(max(proba))
        label_idx  = int(np.argmax(proba))
    else:
        score      = ml_pipeline.decision_function([cleaned])[0]
        confidence = float(abs(score) / (abs(score) + 1))
        label_idx  = 1 if score > 0 else 0

    ml_verdict    = 'Real' if label_idx == 1 else 'Fake'
    ml_confidence = confidence * 100

    result['ml_used']    = True
    result['verdict']    = ml_verdict
    result['confidence'] = round(ml_confidence, 1)

    # Step 4a — Low confidence → Gemini for second opinion
    if confidence < CONFIDENCE_THRESHOLD:
        print(f"[INFO] Low confidence ({ml_confidence:.1f}%). Getting Gemini second opinion.")
        gemini_result = call_gemini(
            build_low_confidence_prompt(article_text, ml_verdict, ml_confidence),
            expect_json=True
        )
        result.update({
            'gemini_used': True,
            'verdict':     gemini_result.get('verdict', ml_verdict),
            'confidence':  gemini_result.get('confidence', ml_confidence),
            'explanation': gemini_result.get('explanation', ''),
            'signals':     gemini_result.get('signals', []),
            'verify_tip':  gemini_result.get('verify_tip', '')
        })

    # Step 4b — High confidence → Gemini for explanation only
    else:
        print(f"[INFO] High confidence ({ml_confidence:.1f}%). Getting Gemini explanation.")
        explanation = call_gemini(
            build_explanation_prompt(
                article_text, ml_verdict, ml_confidence,
                nlp_data['tone_score'], nlp_data['sentiment']
            ),
            expect_json=False
        )
        result.update({
            'gemini_used': True,
            'explanation': explanation,
            'signals':     [],
            'verify_tip':  'Cross-check with Reuters, AP News, or PTI for Indian news.'
        })

    return result


print("Master prediction function defined.")

## Step 4 — End-to-End Test

In [ ]:
import numpy as np

# Test 1: Obvious fake (English)
print("=" * 60)
print("TEST 1: Obvious fake news")
fake_article = "BREAKING!! Scientists CONFIRM vaccines cause autism!! Government hiding truth!! Share before deleted!!"
r1 = full_predict(fake_article)
print(f"  Verdict     : {r1['verdict']}")
print(f"  Confidence  : {r1['confidence']}%")
print(f"  ML used     : {r1['ml_used']}")
print(f"  Gemini used : {r1['gemini_used']}")
print(f"  Explanation : {r1['explanation']}")
print(f"  Tone score  : {r1['nlp']['tone_score']}")

In [ ]:
# Test 2: Real news (English)
print("=" * 60)
print("TEST 2: Real news article")
real_article = """The Reserve Bank of India on Friday kept its benchmark repo rate unchanged 
at 6.5 per cent for the seventh consecutive time, as the six-member Monetary Policy Committee 
voted 5-1 to hold rates, Governor Shaktikanta Das announced."""
r2 = full_predict(real_article)
print(f"  Verdict     : {r2['verdict']}")
print(f"  Confidence  : {r2['confidence']}%")
print(f"  Explanation : {r2['explanation']}")
print(f"  Entities    : {r2['nlp']['entities']}")

In [ ]:
# Test 3: Hindi article (non-English)
print("=" * 60)
print("TEST 3: Hindi article")
hindi_article = """प्रधानमंत्री नरेंद्र मोदी ने आज नई दिल्ली में एक प्रेस कॉन्फ्रेंस में 
देश की आर्थिक स्थिति के बारे में बात की। उन्होंने कहा कि भारत की जीडीपी विकास दर 
इस तिमाही में 7.2 प्रतिशत रही है।"""
r3 = full_predict(hindi_article)
print(f"  Language    : {r3['nlp']['language']}")
print(f"  Verdict     : {r3['verdict']}")
print(f"  Confidence  : {r3['confidence']}%")
print(f"  Explanation : {r3['explanation']}")

## Step 5 — Export Gemini Utils as Python File

In [ ]:
gemini_utils_code = '''
import re
import json
import time
import numpy as np
import pickle
import google.generativeai as genai

def init_gemini(api_key):
    genai.configure(api_key=api_key)
    return genai.GenerativeModel("gemini-1.5-flash")

def call_gemini(model, prompt, expect_json=False, retries=2):
    for attempt in range(retries + 1):
        try:
            response = model.generate_content(
                prompt,
                generation_config=genai.types.GenerationConfig(
                    temperature=0.2, max_output_tokens=400)
            )
            text = response.text.strip()
            if expect_json:
                text = re.sub(r"```json|```", "", text).strip()
                return json.loads(text)
            return text
        except json.JSONDecodeError:
            if attempt < retries:
                time.sleep(1); continue
            return {"verdict": "Uncertain", "confidence": 50,
                    "explanation": "Analysis inconclusive.",
                    "signals": [], "verify_tip": "Check a trusted news source."}
        except Exception as e:
            if attempt < retries:
                time.sleep(2); continue
            return {"error": str(e), "explanation": "Gemini unavailable."}

def get_explanation(model, article_text, ml_verdict, ml_confidence, tone_score, sentiment):
    prompt = f"""
You are VeriFact, an AI fact-checking assistant for students.
ML verdict: {ml_verdict} ({ml_confidence:.1f}% confidence)
Tone score: {tone_score}/100 | Sentiment: {sentiment["label"]} ({sentiment["compound"]})
Article: \"\"\"{article_text[:800]}\"\"\"
Write a 3-4 sentence student-friendly explanation of why this article is {ml_verdict}.
Point out 2-3 specific signals and give one verification tip. English only."""
    return call_gemini(model, prompt, expect_json=False)

def get_multilingual_analysis(model, article_text):
    prompt = f"""
You are VeriFact, a multilingual AI fact-checker for students.
Analyze this article (any language) and respond ONLY in this JSON format:
{{"verdict":"Fake" or "Real","confidence":0-100,"explanation":"2-3 sentences in English",
"signals":["s1","s2","s3"],"verify_tip":"one tip"}}
Article: \"\"\"{article_text[:1200]}\"\"\""""
    return call_gemini(model, prompt, expect_json=True)

def get_low_confidence_analysis(model, article_text, ml_verdict, ml_confidence):
    prompt = f"""
You are VeriFact. ML model was uncertain (verdict: {ml_verdict}, confidence: {ml_confidence:.1f}%).
Give your independent assessment in this JSON format:
{{"verdict":"Fake" or "Real" or "Uncertain","confidence":0-100,"explanation":"2-3 sentences",
"signals":["s1","s2"],"verify_tip":"one tip"}}
Article: \"\"\"{article_text[:1000]}\"\"\""""
    return call_gemini(model, prompt, expect_json=True)
'''

save_path = '/content/drive/MyDrive/VeriFact/utils/gemini_utils.py'
with open(save_path, 'w') as f:
    f.write(gemini_utils_code.strip())

print(f"gemini_utils.py saved to: {save_path}")
print("\nNotebook 4 complete. Proceed to building the Streamlit app.")

## Summary

| Use Case | Prompt Type | JSON? |
|---|---|---|
| High confidence English | Explanation | No (free text) |
| Low confidence English | Second opinion | Yes |
| Non-English article | Full analysis | Yes |

**Next step →** Build the Streamlit app (`app/app.py`)